## Import relevant packages

In [35]:
import pandas as pd
import numpy as np
import math

In [36]:
def format_engineering(val, precision=3):
    """
    Formats a numeric value into true engineering notation.
    Numerically zero values output a clean, uniform string.
    """
    try:
        val_float = float(val)
    except (ValueError, TypeError):
        return str(val)
    if abs(val_float) < 1e-9:
        return "0"
        
    sign = "-" if val_float < 0 else ""
    val_abs = abs(val_float)
    
    exp = int(math.floor(math.log10(val_abs) / 3.0) * 3)
    mantissa = val_abs / (10**exp)
    
    if round(mantissa, precision) >= 1000.0:
        mantissa /= 1000.0
        exp += 3
        
    return f"{sign}{mantissa:.{precision}f}e{exp:+03d}"

## Read structure properties.csv

In [37]:
# 1. Read the raw properties CSV file
df = pd.read_csv('properties.csv')

# 2. Isolate the data frames for calculations
df_geom = df[['Member_ID', 'Node_I', 'Node_J', 'Xi', 'Yi', 'Zi', 'Xj', 'Yj', 'Zj']]
df_props = df[['Member_ID', 'L (mm)', 'Area (mm2)', 'E (GPa)', 'v', 'J (mm4)', 'Iz (mm4)', 'Iy (mm4)', 'Omega (deg)']]

In [38]:
df_geom # geometry

,Member_ID,Node_I,Node_J,Xi,Yi,Zi,Xj,Yj,Zj
0,1,1,4,0,0,0,0,1500,0
1,2,2,5,0,0,4500,0,1500,4500
2,3,3,6,0,0,8150,0,1500,8150
3,4,24,28,4800,0,0,4800,1500,0
4,5,25,29,4800,0,4500,4800,1500,4500
...,...,...,...,...,...,...,...,...,...
187,188,90,110,7630,12500,3500,7630,12500,6325
188,189,110,93,7630,12500,6325,7630,12500,8150
189,190,84,111,11000,13300,4500,11000,13300,6325
190,191,111,85,11000,13300,6325,11000,13300,8150


In [39]:
df_props # section properties

,Member_ID,L (mm),Area (mm2),E (GPa),v,J (mm4),Iz (mm4),Iy (mm4),Omega (deg)
0,1,1500.00,100000,21.5,0.3,2.416667e+09,2083333333,333333333.0,0
1,2,1500.00,100000,21.5,0.3,2.416667e+09,2083333333,333333333.0,0
2,3,1500.00,100000,21.5,0.3,2.416667e+09,2083333333,333333333.0,0
3,4,1500.00,100000,21.5,0.3,2.416667e+09,2083333333,333333333.0,0
4,5,1500.00,125000,21.5,0.3,6.510417e+08,2604166667,651041666.7,0
...,...,...,...,...,...,...,...,...,...
187,188,2825.00,80000,21.5,0.3,1.333333e+09,1066666667,266666666.7,0
188,189,1825.00,80000,21.5,0.3,1.333333e+09,1066666667,266666666.7,0
189,190,1825.00,80000,21.5,0.3,1.333333e+09,1066666667,266666666.7,0
190,191,1825.00,80000,21.5,0.3,1.333333e+09,1066666667,266666666.7,0


## Build local element stiffness matrix [k], rotation matrix [r], and global element stiffness matrix [k]

In [40]:
# ==============================================================================
# 1. INITIALIZE DATA & STORAGE LISTS
# ==============================================================================
df = pd.read_csv('properties.csv')

max_member_id = int(df['Member_ID'].max())

list_k_local  = [None] * (max_member_id + 1)
list_r        = [None] * (max_member_id + 1)
list_k_global = [None] * (max_member_id + 1)

# ==============================================================================
# 2. MATRIX GENERATION LOOP WITH DYNAMIC NODE LABELS
# ==============================================================================
for index, row in df.iterrows():
    mem_id = int(row['Member_ID'])
    node_i = int(row['Node_I'])
    node_j = int(row['Node_J'])
    
    # Generate dynamic, node-specific DOF labels for this specific element
    dof_labels_row = [
        f'Fx{node_i}',  f'Fy{node_i}',  f'Fz{node_i}',  f'Mx{node_i}',  f'My{node_i}',  f'Mz{node_i}',
        f'Fx{node_j}',  f'Fy{node_j}',  f'Fz{node_j}',  f'Mx{node_j}',  f'My{node_j}',  f'Mz{node_j}'
    ]
    dof_labels_col = [
        f'u{node_i}',  f'v{node_i}',  f'w{node_i}',  f'θx{node_i}',  f'θy{node_i}',  f'θz{node_i}',
        f'u{node_j}',  f'v{node_j}',  f'w{node_j}',  f'θx{node_j}',  f'θy{node_j}',  f'θz{node_j}'
    ]
    
    # Extract properties & enforce unit consistency (kN and mm)
    L  = float(row['L (mm)'])
    A  = float(row['Area (mm2)'])
    
    # 1 GPa = 1 kN/mm^2. No conversion required.
    E  = float(row['E (GPa)']) 
    G  = E / (2.0 * (1.0 + float(row['v'])))
    
    J  = float(row['J (mm4)'])
    Iy = float(row['Iy (mm4)'])
    Iz = float(row['Iz (mm4)'])
    omega = np.radians(float(row['Omega (deg)'])) # Input in deg, calculate in rad
    
    # Coordinate extraction
    xi, yi, zi = float(row['Xi']), float(row['Yi']), float(row['Zi'])
    xj, yj, zj = float(row['Xj']), float(row['Yj']), float(row['Zj'])

    # --------------------------------------------------------------------------
    # A. BUILD 12x12 LOCAL STIFFNESS MATRIX [k_local]
    # --------------------------------------------------------------------------
    k_loc = np.zeros((12, 12))
    
    # Axial Terms
    k_loc[0,0] = k_loc[6,6] =  A*E/L
    k_loc[0,6] = k_loc[6,0] = -A*E/L
    
    # Torsional Terms
    k_loc[3,3] = k_loc[9,9] =  G*J/L
    k_loc[3,9] = k_loc[9,3] = -G*J/L
    
    # Bending about Local Z-Axis (Deflection in Local Y-Direction)
    k_loc[1,1]   = k_loc[7,7]   =  12*E*Iz / (L**3)
    k_loc[1,7]   = k_loc[7,1]   = -12*E*Iz / (L**3)
    k_loc[1,5]   = k_loc[1,11]  = k_loc[5,1]  = k_loc[11,1]  =  6*E*Iz / (L**2)
    k_loc[5,7]   = k_loc[7,5]   = k_loc[7,11] = k_loc[11,7] = -6*E*Iz / (L**2)
    k_loc[5,5]   = k_loc[11,11] =  4*E*Iz / L
    k_loc[5,11]  = k_loc[11,5]  =  2*E*Iz / L
    
    # Bending about Local Y-Axis (Deflection in Local Z-Direction)
    k_loc[2,2]   = k_loc[8,8]   =  12*E*Iy / (L**3)
    k_loc[2,8]   = k_loc[8,2]   = -12*E*Iy / (L**3)
    k_loc[2,4]   = k_loc[2,10]  = k_loc[4,2]  = k_loc[10,2]  = -6*E*Iy / (L**2)
    k_loc[4,8]   = k_loc[8,4]   = k_loc[8,10] = k_loc[10,8] =  6*E*Iy / (L**2)
    k_loc[4,4]   = k_loc[10,10] =  4*E*Iy / L
    k_loc[4,10]  = k_loc[10,4]  =  2*E*Iy / L

    # --------------------------------------------------------------------------
    # B. BUILD 3x3 DIRECTIONAL COSINE MATRIX [R] USING VECTORS
    # --------------------------------------------------------------------------
    vec_x = np.array([xj - xi, yj - yi, zj - zi]) / L
    
    if np.isclose(vec_x[0], 0.0, atol=1e-6) and np.isclose(vec_x[2], 0.0, atol=1e-6):
        if vec_x[1] > 0: 
            vec_y0 = np.array([-1.0,  0.0,  0.0])
            vec_z0 = np.array([ 0.0,  0.0,  1.0])
        else:            
            vec_y0 = np.array([ 1.0,  0.0,  0.0])
            vec_z0 = np.array([ 0.0,  0.0,  1.0])
    else:
        global_Y = np.array([0.0, 1.0, 0.0])
        cross_z = np.cross(vec_x, global_Y)
        vec_z0 = cross_z / np.linalg.norm(cross_z)
        vec_y0 = np.cross(vec_z0, vec_x)
        
    vec_y = vec_y0 * np.cos(omega) + vec_z0 * np.sin(omega)
    vec_z = -vec_y0 * np.sin(omega) + vec_z0 * np.cos(omega)
    
    R_block = np.vstack([vec_x, vec_y, vec_z])
    
    T_mat = np.zeros((12, 12))
    for i in range(4):
        T_mat[i*3:(i+1)*3, i*3:(i+1)*3] = R_block

    # --------------------------------------------------------------------------
    # C. TRANSFORM TO GLOBAL STIFFNESS MATRIX
    # --------------------------------------------------------------------------
    k_glob = T_mat.T @ k_loc @ T_mat
    
    # --------------------------------------------------------------------------
    # D. SAVE TO LISTS WITH CUSTOM ELEMENT DOF LABELS
    # --------------------------------------------------------------------------
    list_k_local[mem_id]  = pd.DataFrame(k_loc,  index=dof_labels_row, columns=dof_labels_col)
    list_r[mem_id]        = pd.DataFrame(T_mat,  index=dof_labels_row, columns=dof_labels_col)
    list_k_global[mem_id] = pd.DataFrame(k_glob, index=dof_labels_row, columns=dof_labels_col)

print("Matrices generated in kN, mm, and rad successfully.")

Matrices generated in kN, mm, and rad successfully.


In [41]:
list_k_local[1].map(lambda x: format_engineering(x)) # [k'] local member stiffnes matrix of Member_ID

,u1,v1,w1,θx1,θy1,θz1,u4,v4,w4,θx4,θy4,θz4
Fx1,1.433e+03,0,0,0,0,0,-1.433e+03,0,0,0,0,0
Fy1,0,159.259e+00,0,0,0,119.444e+03,0,-159.259e+00,0,0,0,119.444e+03
Fz1,0,0,25.481e+00,0,-19.111e+03,0,0,0,-25.481e+00,0,-19.111e+03,0
Mx1,0,0,0,13.323e+06,0,0,0,0,0,-13.323e+06,0,0
My1,0,0,-19.111e+03,0,19.111e+06,0,0,0,19.111e+03,0,9.556e+06,0
Mz1,0,119.444e+03,0,0,0,119.444e+06,0,-119.444e+03,0,0,0,59.722e+06
Fx4,-1.433e+03,0,0,0,0,0,1.433e+03,0,0,0,0,0
Fy4,0,-159.259e+00,0,0,0,-119.444e+03,0,159.259e+00,0,0,0,-119.444e+03
Fz4,0,0,-25.481e+00,0,19.111e+03,0,0,0,25.481e+00,0,19.111e+03,0
Mx4,0,0,0,-13.323e+06,0,0,0,0,0,13.323e+06,0,0


In [42]:
list_r[1]  # [r] rotation matrix of Member_ID

,u1,v1,w1,θx1,θy1,θz1,u4,v4,w4,θx4,θy4,θz4
Fx1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Fy1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Fz1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mx1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
My1,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mz1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
Fx4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Fy4,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0
Fz4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
Mx4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [43]:
list_k_global[1].map(lambda x: format_engineering(x)) # [k] global element stiffness matrix of Member_ID

,u1,v1,w1,θx1,θy1,θz1,u4,v4,w4,θx4,θy4,θz4
Fx1,159.259e+00,0,0,0,0,-119.444e+03,-159.259e+00,0,0,0,0,-119.444e+03
Fy1,0,1.433e+03,0,0,0,0,0,-1.433e+03,0,0,0,0
Fz1,0,0,25.481e+00,19.111e+03,0,0,0,0,-25.481e+00,19.111e+03,0,0
Mx1,0,0,19.111e+03,19.111e+06,0,0,0,0,-19.111e+03,9.556e+06,0,0
My1,0,0,0,0,13.323e+06,0,0,0,0,0,-13.323e+06,0
Mz1,-119.444e+03,0,0,0,0,119.444e+06,119.444e+03,0,0,0,0,59.722e+06
Fx4,-159.259e+00,0,0,0,0,119.444e+03,159.259e+00,0,0,0,0,119.444e+03
Fy4,0,-1.433e+03,0,0,0,0,0,1.433e+03,0,0,0,0
Fz4,0,0,-25.481e+00,-19.111e+03,0,0,0,0,25.481e+00,-19.111e+03,0,0
Mx4,0,0,19.111e+03,9.556e+06,0,0,0,0,-19.111e+03,19.111e+06,0,0


## Global stiffness matrix [K]

In [44]:
# ==============================================================================
# 1. DETERMINE MASTER SYSTEM SIZE & LABELS
# ==============================================================================
# Find the highest node ID to determine the total size of the system matrix
max_node_id = int(max(df['Node_I'].max(), df['Node_J'].max()))
dofs_per_node = 6
total_system_dofs = max_node_id * dofs_per_node

# Generate the sequential DOF labels for the master matrix (u1, v1 ... θz_max)
master_labels = []
for n in range(1, max_node_id + 1):
    master_labels.extend([f'u{n}', f'v{n}', f'w{n}', f'θx{n}', f'θy{n}', f'θz{n}'])

# Initialize the master structural stiffness array with zeros
K_master_array = np.zeros((total_system_dofs, total_system_dofs))

# ==============================================================================
# 2. ASSEMBLE MATRICES (Direct Stiffness Method)
# ==============================================================================
for index, row in df.iterrows():
    mem_id = int(row['Member_ID'])
    node_i = int(row['Node_I'])
    node_j = int(row['Node_J'])
    
    # Extract the underlying numpy array from the DataFrame in your list
    k_element_global = list_k_global[mem_id].values
    
    # Calculate the exact 0-based index positions in the master matrix
    # Node I mapping (6 DOFs)
    dofs_i = [(node_i - 1) * dofs_per_node + r for r in range(dofs_per_node)]
    # Node J mapping (6 DOFs)
    dofs_j = [(node_j - 1) * dofs_per_node + r for r in range(dofs_per_node)]
    
    # Combine to form the 12-element mapping vector
    element_dof_mapping = dofs_i + dofs_j
    
    # Scatter and accumulate the element matrix into the master matrix
    # Using np.ix_ creates an open index mesh, bypassing the need for nested loops
    K_master_array[np.ix_(element_dof_mapping, element_dof_mapping)] += k_element_global

# ==============================================================================
# 3. CONVERT TO DATAFRAME
# ==============================================================================
df_K_sys = pd.DataFrame(K_master_array, index=master_labels, columns=master_labels)

print("Master System Stiffness Matrix Assembly Complete!")
print(f"Total Matrix Size: {df_K_sys.shape[0]} x {df_K_sys.shape[1]}")

Master System Stiffness Matrix Assembly Complete!
Total Matrix Size: 666 x 666


In [45]:
df_K_sys.map(lambda x: format_engineering(x)) # [K] structure stiffness matrix

,u1,v1,w1,θx1,θy1,θz1,u2,v2,w2,θx2,...,w110,θx110,θy110,θz110,u111,v111,w111,θx111,θy111,θz111
u1,518.348e+00,0,0,0,1.699e+03,-119.444e+03,-755.007e-03,0,0,0,...,0,0,0,0,0,0,0,0,0,0
v1,0,1.439e+03,0,-6.795e+03,0,5.972e+03,0,-3.020e+00,0,-6.795e+03,...,0,0,0,0,0,0,0,0,0,0
w1,0,0,408.326e+00,19.111e+03,-1.493e+03,0,0,0,-382.222e+00,0,...,0,0,0,0,0,0,0,0,0,0
θx1,0,-6.795e+03,19.111e+03,41.793e+06,0,0,0,6.795e+03,0,10.193e+06,...,0,0,0,0,0,0,0,0,0,0
θy1,1.699e+03,0,-1.493e+03,0,23.197e+06,0,-1.699e+03,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
v111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,-11.160e+03,110.107e+00,123.311e+00,0,0,0,-11.160e+03
w111,0,0,0,0,0,0,0,0,0,0,...,-1.656e+00,-662.287e+00,2.790e+03,0,0,0,1.887e+03,-662.287e+00,2.790e+03,0
θx111,0,0,0,0,0,0,0,0,0,0,...,662.287e+00,-2.837e+06,-1.459e+06,0,0,0,-662.287e+00,103.896e+06,-772.584e+03,0
θy111,0,0,0,0,0,0,0,0,0,0,...,-2.790e+03,-1.459e+06,2.964e+06,0,0,0,2.790e+03,-772.584e+03,31.570e+06,0


## Read node fixities.csv. Get [K_ff], [K_sf], [K_fs], [K_ss]

In [46]:
# ==============================================================================
# 1. LOAD FIXITIES DATAFRAME
# ==============================================================================
df_fixities = pd.read_csv('fixities.csv')

# Clean up column names in case there are accidental spaces in the CSV
df_fixities.columns = df_fixities.columns.str.strip()

print("--- Nodal Fixities ---")
display(df_fixities)

# ==============================================================================
# 2. CLASSIFY DEGREES OF FREEDOM (Free vs Supported)
# ==============================================================================
dof_f = [] # List for Free DOFs
dof_s = [] # List for Supported/Fixed DOFs

# We loop through all nodes in the structural system
# (Assuming max_node_id is still defined from your master assembly step)
for n in range(1, max_node_id + 1):
    
    # Check if this node has defined fixities in the CSV
    node_data = df_fixities[df_fixities['node'] == n]
    
    if not node_data.empty:
        # Extract the string values and convert to lowercase to avoid case-sensitivity issues
        fix_u  = str(node_data['x'].values[0]).strip().lower()
        fix_v  = str(node_data['y'].values[0]).strip().lower()
        fix_w  = str(node_data['z'].values[0]).strip().lower()
        fix_rx = str(node_data['x-rot'].values[0]).strip().lower()
        fix_ry = str(node_data['y-rot'].values[0]).strip().lower()
        fix_rz = str(node_data['z-rot'].values[0]).strip().lower()
    else:
        # If a node is missing from the CSV, assume it is completely free in space
        fix_u = fix_v = fix_w = fix_rx = fix_ry = fix_rz = 'free'

    # Map the fixity states to your specific nodal DOF labels
    dof_mappings = [
        (fix_u,  f'u{n}'), 
        (fix_v,  f'v{n}'), 
        (fix_w,  f'w{n}'), 
        (fix_rx, f'θx{n}'), 
        (fix_ry, f'θy{n}'), 
        (fix_rz, f'θz{n}')
    ]
    
    # Sort into the correct lists
    for state, dof_label in dof_mappings:
        if state == 'fixed':
            dof_s.append(dof_label)
        else:
            dof_f.append(dof_label)

print(f"\nTotal Free DOFs (f): {len(dof_f)}")
print(f"Total Supported DOFs (s): {len(dof_s)}")

# ==============================================================================
# 3. PARTITION THE MASTER STIFFNESS MATRIX
# ==============================================================================
# Using Pandas .loc[], we can effortlessly slice the assembled df_K_sys using our categorized lists

# [Kff]: Free-to-Free stiffness (Used to solve for unknown displacements)
df_Kff = df_K_sys.loc[dof_f, dof_f]

# [Kfs]: Free-to-Supported stiffness
df_Kfs = df_K_sys.loc[dof_f, dof_s]

# [Ksf]: Supported-to-Free stiffness (Used to solve for support reactions)
df_Ksf = df_K_sys.loc[dof_s, dof_f]

# [Kss]: Supported-to-Supported stiffness
df_Kss = df_K_sys.loc[dof_s, dof_s]

# ==============================================================================
# 4. VERIFICATION PRINTOUT
# ==============================================================================
print("\n--- MATRIX PARTITION SIZES ---")
print(f"[Kff] Size: {df_Kff.shape[0]} x {df_Kff.shape[1]}")
print(f"[Kfs] Size: {df_Kfs.shape[0]} x {df_Kfs.shape[1]}")
print(f"[Ksf] Size: {df_Ksf.shape[0]} x {df_Ksf.shape[1]}")
print(f"[Kss] Size: {df_Kss.shape[0]} x {df_Kss.shape[1]}")

--- Nodal Fixities ---


,node,x,y,z,x-rot,y-rot,z-rot
0,1,fixed,fixed,fixed,fixed,fixed,fixed
1,2,fixed,fixed,fixed,fixed,fixed,fixed
2,3,fixed,fixed,fixed,fixed,fixed,fixed
3,24,fixed,fixed,fixed,fixed,fixed,fixed
4,25,fixed,fixed,fixed,fixed,fixed,fixed
5,26,fixed,fixed,fixed,fixed,fixed,fixed
6,27,fixed,fixed,fixed,fixed,fixed,fixed
7,46,fixed,fixed,fixed,fixed,fixed,fixed
8,47,fixed,fixed,fixed,fixed,fixed,fixed
9,62,fixed,fixed,fixed,fixed,fixed,fixed



Total Free DOFs (f): 600
Total Supported DOFs (s): 66

--- MATRIX PARTITION SIZES ---
[Kff] Size: 600 x 600
[Kfs] Size: 600 x 66
[Ksf] Size: 66 x 600
[Kss] Size: 66 x 66


In [47]:
df_Kff.map(lambda x: format_engineering(x))

,u4,v4,w4,θx4,θy4,θz4,u5,v5,w5,θx5,...,w110,θx110,θy110,θz110,u111,v111,w111,θx111,θy111,θz111
u4,181.298e+00,0,0,0,0,87.488e+03,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
v4,0,2.175e+03,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
w4,0,0,29.008e+00,-13.998e+03,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θx4,0,0,-13.998e+03,28.996e+06,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θy4,0,0,0,0,20.214e+06,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
v111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,-11.160e+03,110.107e+00,123.311e+00,0,0,0,-11.160e+03
w111,0,0,0,0,0,0,0,0,0,0,...,-1.656e+00,-662.287e+00,2.790e+03,0,0,0,1.887e+03,-662.287e+00,2.790e+03,0
θx111,0,0,0,0,0,0,0,0,0,0,...,662.287e+00,-2.837e+06,-1.459e+06,0,0,0,-662.287e+00,103.896e+06,-772.584e+03,0
θy111,0,0,0,0,0,0,0,0,0,0,...,-2.790e+03,-1.459e+06,2.964e+06,0,0,0,2.790e+03,-772.584e+03,31.570e+06,0


In [48]:
df_Kfs.map(lambda x: format_engineering(x))

,u1,v1,w1,θx1,θy1,θz1,u2,v2,w2,θx2,...,w62,θx62,θy62,θz62,u63,v63,w63,θx63,θy63,θz63
u4,-159.259e+00,0,0,0,0,119.444e+03,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
v4,0,-1.433e+03,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
w4,0,0,-25.481e+00,-19.111e+03,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θx4,0,0,19.111e+03,9.556e+06,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θy4,0,0,0,0,-13.323e+06,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
v111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
w111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θx111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θy111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
df_Ksf.map(lambda x: format_engineering(x))

,u4,v4,w4,θx4,θy4,θz4,u5,v5,w5,θx5,...,w110,θx110,θy110,θz110,u111,v111,w111,θx111,θy111,θz111
u1,-159.259e+00,0,0,0,0,-119.444e+03,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
v1,0,-1.433e+03,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
w1,0,0,-25.481e+00,19.111e+03,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θx1,0,0,-19.111e+03,9.556e+06,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θy1,0,0,0,0,-13.323e+06,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
v63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
w63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θx63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
θy63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [50]:
df_Kss.map(lambda x: format_engineering(x))

,u1,v1,w1,θx1,θy1,θz1,u2,v2,w2,θx2,...,w62,θx62,θy62,θz62,u63,v63,w63,θx63,θy63,θz63
u1,518.348e+00,0,0,0,1.699e+03,-119.444e+03,-755.007e-03,0,0,0,...,0,0,0,0,0,0,0,0,0,0
v1,0,1.439e+03,0,-6.795e+03,0,5.972e+03,0,-3.020e+00,0,-6.795e+03,...,0,0,0,0,0,0,0,0,0,0
w1,0,0,408.326e+00,19.111e+03,-1.493e+03,0,0,0,-382.222e+00,0,...,0,0,0,0,0,0,0,0,0,0
θx1,0,-6.795e+03,19.111e+03,41.793e+06,0,0,0,6.795e+03,0,10.193e+06,...,0,0,0,0,0,0,0,0,0,0
θy1,1.699e+03,0,-1.493e+03,0,23.197e+06,0,-1.699e+03,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
v63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,2.153e+03,0,0,0,-12.087e+03
w63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,87.003e+00,64.500e+03,4.113e+03,0
θx63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,64.500e+03,73.015e+06,0,0
θy63,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,4.113e+03,0,45.913e+06,0


## Read slab loads and self-weight loads as nodal forces and support displacements

In [51]:
# ==============================================================================
# 1. LOAD AND COMBINE NODAL LOADS
# ==============================================================================
# Read the slab loads and self-weight loads
df_loads_slab = pd.read_csv('loads_slab.csv')
df_loads_sw = pd.read_csv('loads_sw.csv')

# Clean headers to ensure no whitespace issues
df_loads_slab.columns = df_loads_slab.columns.str.strip()
df_loads_sw.columns = df_loads_sw.columns.str.strip()

# Combine the loads by adding them together grouped by node
df_loads_total = pd.concat([df_loads_slab, df_loads_sw]).groupby('node', as_index=False).sum()

# Initialize the full load vector {P} with zeros
P_full = pd.Series(0.0, index=master_labels)

# Map the combined CSV loads into the full {P} vector
for _, row in df_loads_total.iterrows():
    n = int(row['node'])
    P_full[f'u{n}']  = row['Fx']
    P_full[f'v{n}']  = row['Fy']
    P_full[f'w{n}']  = row['Fz']
    P_full[f'θx{n}'] = row['Mx']
    P_full[f'θy{n}'] = row['My']
    P_full[f'θz{n}'] = row['Mz']

# Extract the Free Loads {Pf} using the dof_f list
P_f = P_full[dof_f].values

# ==============================================================================
# 2. LOAD SUPPORT DISPLACEMENTS {Ds}
# ==============================================================================
# Initialize the full displacement vector {D} with zeros
D_full = pd.Series(0.0, index=master_labels)

try:
    df_supp_disp = pd.read_csv('support_disp.csv')
    df_supp_disp.columns = df_supp_disp.columns.str.strip()
    
    # Map the known support displacements into the {D} vector
    for _, row in df_supp_disp.iterrows():
        n = int(row['node'])
        D_full[f'u{n}']  = row['ui']
        D_full[f'v{n}']  = row['vi']
        D_full[f'w{n}']  = row['wi']
        D_full[f'θx{n}'] = row['θxi']
        D_full[f'θy{n}'] = row['θyi']
        D_full[f'θz{n}'] = row['θzi']
except FileNotFoundError:
    print("No 'support_disp.csv' found. Assuming all support settlements = 0.0\n")

# Extract the Known Supported Displacements {Ds}
D_s = D_full[dof_s].values

No 'support_disp.csv' found. Assuming all support settlements = 0.0



In [52]:
df_loads_total

,node,Floor,Archi_Node,Fx,Fy,Fz,Mx,My,Mz
0,1,0,0,0,-12.528000,0,3240.000000,0.000000,-3686.400000
1,2,0,0,0,-16.032000,0,-1108.400000,0.000000,-3686.400000
2,3,0,0,0,-11.712000,0,-2131.600000,0.000000,-3686.400000
3,4,0,0,0,-6.960000,0,0.000000,0.000000,0.000000
4,5,0,0,0,-6.960000,0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
106,107,roof,B-3',0,-17.989489,0,-3253.994155,1433.649493,-5350.885670
107,108,roof,B1-3',0,-18.949489,0,-3997.994155,1433.649493,5350.885670
108,109,roof,D-2A,0,-13.966210,0,-2752.461766,-419.636995,3829.490779
109,110,roof,C1-3',0,-13.232278,0,-374.799332,162.662969,-3792.797664


## Solve for free nodes global deformations

In [53]:
# ==============================================================================
# 3. SOLVE THE SYSTEM OF EQUATIONS
# ==============================================================================
Kff_matrix = df_Kff.values
Kfs_matrix = df_Kfs.values

# Calculate the effective load vector: {Pf*} = {Pf} - [Kfs]{Ds}
P_f_effective = P_f - (Kfs_matrix @ D_s)

# Solve for unknown displacements: [Kff]{Df} = {Pf*}
if len(dof_f) > 0:
    D_f_solved = np.linalg.solve(Kff_matrix, P_f_effective)
else:
    D_f_solved = np.array([])

# Map the solved free displacements back into the master {D} vector
# (D_full now contains BOTH the solved {Df} and the prescribed {Ds})
D_full[dof_f] = D_f_solved

# ==============================================================================
# 4. RESTRUCTURE RESULTS INTO A COMPLETE DATAFRAME
# ==============================================================================
# Generate a list of all nodes from 1 to max_node_id
all_nodes = list(range(1, max_node_id + 1))

# Build the structured DataFrame for mathematical storage (for all nodes)
df_delta_all = pd.DataFrame(index=all_nodes, columns=['ui', 'vi', 'wi', 'θxi', 'θyi', 'θzi'])
df_delta_all.index.name = "Node"

for n in all_nodes:
    df_delta_all.loc[n, 'ui']  = D_full[f'u{n}']
    df_delta_all.loc[n, 'vi']  = D_full[f'v{n}']
    df_delta_all.loc[n, 'wi']  = D_full[f'w{n}']
    df_delta_all.loc[n, 'θxi'] = D_full[f'θx{n}']
    df_delta_all.loc[n, 'θyi'] = D_full[f'θy{n}']
    df_delta_all.loc[n, 'θzi'] = D_full[f'θz{n}']

# Create a formatted copy for display using your engineering format function
# Note: modern pandas prefers .map() or .apply(), using map for backwards compatibility
df_delta_all_display = df_delta_all.map(lambda x: format_engineering(x))


In [54]:
df_delta_all.map(lambda x: format_engineering(x)) #{delta_F} deformation of free nodes

,ui,vi,wi,θxi,θyi,θzi
Node,,,,,,
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,45.259e-03,-78.408e-03,72.094e-03,98.933e-06,-5.004e-06,-60.689e-06
5,-49.194e-03,-109.790e-03,216.932e-03,242.354e-06,-499.553e-09,47.497e-06
...,...,...,...,...,...,...
107,1.919e+00,-869.531e-03,2.046e+00,-96.687e-06,68.906e-06,-493.963e-06
108,1.683e+00,-1.989e+00,2.970e+00,-368.305e-06,133.013e-06,-162.713e-06
109,499.422e-03,-676.344e-03,5.037e+00,82.467e-06,-183.676e-06,963.188e-06


In [55]:
df_delta_all.to_clipboard()

## Solve for Reactions

In [56]:
# ==============================================================================
# 1. EXTRACT MATRICES AND VECTORS
# ==============================================================================
# Convert the Pandas submatrices to NumPy arrays for calculation
Ksf_matrix = df_Ksf.values
Kss_matrix = df_Kss.values

# Extract the solved free deformations {Df} and known support displacements {Ds}
# (D_full was fully assembled in the previous deformation solving step)
D_f = D_full[dof_f].values
D_s = D_full[dof_s].values

# ==============================================================================
# 2. EVALUATE FORCES AT SUPPORTED DEGREES OF FREEDOM
# ==============================================================================
# Calculate the total internal nodal forces required for equilibrium at the supports
# {Ps} = [Ksf]{Df} + [Kss]{Ds}
P_s_internal = (Ksf_matrix @ D_f) + (Kss_matrix @ D_s)

# Extract any external loads that were directly applied to the supported DOFs
# (P_full already contains the sum of the slab and self-weight CSV loads)
P_s_applied = P_full[dof_s].values

# Calculate the True Support Reactions {Rs}
# The physical support must resist BOTH the structure's internal forces AND 
# any external load pushing directly against the boundary.
# {Rs} = {Ps_internal} - {Ps_applied}
R_s = P_s_internal - P_s_applied

# ==============================================================================
# 3. RESTRUCTURE RESULTS INTO A DATAFRAME
# ==============================================================================
# Map the calculated reactions back into a master series for easy extraction
R_full = pd.Series(0.0, index=master_labels)
R_full[dof_s] = R_s

# Identify which nodes actually have at least one supported DOF
# (We parse the node IDs out of the dof_s text labels)
supported_nodes = sorted(list(set([int(dof.replace('u','').replace('v','').replace('w','')
                                 .replace('θx','').replace('θy','').replace('θz','')) 
                              for dof in dof_s])))

# Build the structured DataFrame for mathematical storage
df_reactions = pd.DataFrame(index=supported_nodes, columns=['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz'])
df_reactions.index.name = "Node"

for n in supported_nodes:
    # If a specific DOF at a supported node was actually 'free', R_full will 
    # correctly yield 0.0 because it wasn't populated by the R_s vector.
    df_reactions.loc[n, 'Fx'] = R_full[f'u{n}']
    df_reactions.loc[n, 'Fy'] = R_full[f'v{n}']
    df_reactions.loc[n, 'Fz'] = R_full[f'w{n}']
    df_reactions.loc[n, 'Mx'] = R_full[f'θx{n}']
    df_reactions.loc[n, 'My'] = R_full[f'θy{n}']
    df_reactions.loc[n, 'Mz'] = R_full[f'θz{n}']


In [57]:
df_reactions.map(lambda x: format_engineering(x)) # {Ps} Reactions of support nodes

,Fx,Fy,Fz,Mx,My,Mz
Node,,,,,,
1,40.965e-03,124.913e+00,53.673e-03,-3.672e+03,66.665e+00,5.468e+03
2,2.161e+00,173.398e+00,-896.099e-03,-721.588e+00,6.655e+00,647.089e+00
3,2.291e+00,111.584e+00,-703.992e-03,580.199e+00,-39.207e+00,-1.214e+03
24,-448.490e-03,256.594e+00,875.916e-03,-2.054e+03,-31.605e+00,5.002e+03
25,2.012e+00,368.705e+00,-309.186e-03,2.591e+03,51.457e+00,-259.533e+00
26,6.956e+00,426.244e+00,-2.134e+00,-3.629e+03,869.405e+00,-16.580e+03
27,31.029e+00,122.796e+00,-2.397e+00,-3.830e+03,307.212e+00,-22.899e+03
46,-3.364e+00,136.283e+00,144.467e-03,-3.117e+03,246.930e+00,747.420e+00
47,-4.353e+00,366.774e+00,8.117e+00,14.859e+03,77.643e+00,643.571e+00


In [58]:
df_reactions.to_clipboard()

## Solve for local member forces

In [59]:
def evaluate_local_member_forces(
    df_geom, 
    list_k_local, 
    list_r, 
    df_delta_all, 
    df_loads_total
):
    """
    Re-evaluates the local member forces and stores all intermediate configurations
    and state vectors into structured lists of dataframes and a summary dataframe.
    
    Formula tracking:
      {u_local} = [r] * {U_global}
      {f_fixed_local} = [r] * {F_fixed_global}
      {f_local} = [k_local] * {u_local} - {f_fixed_local}
    """
    # 1. Initialize output containers matching structural tracking specifications
    list_u_local_df = {}      # {local displacement matrix} per element
    list_f_fixed_local_df = {} # {local fixed end forces} per element
    list_f_local_df = {}       # {local member forces} per element
    element_force_summary = []
    
    # Track element checklist directly from the geometry configuration
    if 'Member_ID' in df_geom.columns:
        element_ids = df_geom['Member_ID'].tolist()
    else:
        element_ids = df_geom.index.tolist()
        
    # Pre-allocate dictionary mapping to assemble the absolute local force accumulation
    # Across all elements mapped to their specific structural nodes.
    node_force_accumulator = {}

    for elem_id in element_ids:
        # 2. Extract topology connectivity tracking for the current element
        if 'Member_ID' in df_geom.columns:
            geom_row = df_geom[df_geom['Member_ID'] == elem_id].iloc[0]
        else:
            geom_row = df_geom.loc[elem_id]
            
        node_i = int(geom_row['Node_I'])
        node_j = int(geom_row['Node_J'])
        
        # 3. Retrieve baseline matrices (handling both dict tracking and list alignment)
        if isinstance(list_k_local, dict):
            k_local_df = list_k_local[elem_id]
            r_matrix_df = list_r[elem_id]
        else:
            # Map index tracking handling 0-based lookups cleanly
            idx = elem_id if elem_id < len(list_k_local) else elem_id - 1
            k_local_df = list_k_local[idx]
            r_matrix_df = list_r[idx]
            
        # Extract explicit index and column arrays from the structural matrices
        dof_cols = r_matrix_df.columns.tolist() # e.g., ['u2','v2','w2','θx2',...]
        force_rows = r_matrix_df.index.tolist() # e.g., ['Fx2','Fy2','Fz2','Mx2',...]
        
        # Convert dataframes directly to raw numpy arrays for optimized computation
        k_local = k_local_df.to_numpy()
        r_matrix = r_matrix_df.to_numpy()
        
        # ======================================================================
        # STEP A: EXTRACT & COMPUTE GLOBAL STATE VECTORS ({U_global} & {F_fixed_global})
        # ======================================================================
        U_global = np.zeros(12)
        F_fixed_global = np.zeros(12)
        
        # Mapping mapping keys for standardizing df_delta_all lookups
        dof_mapping = ['ui', 'vi', 'wi', 'θxi', 'θyi', 'θzi']
        load_cols = ['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']
        
        # Slice Node I structural global state inputs
        if node_i in df_delta_all.index:
            U_global[0:6] = df_delta_all.loc[node_i, dof_mapping].to_numpy().astype(float)
            
        node_i_loads = df_loads_total[df_loads_total['node'] == node_i]
        if not node_i_loads.empty:
            F_fixed_global[0:6] = node_i_loads[load_cols].iloc[0].to_numpy().astype(float)
            
        # Slice Node J structural global state inputs
        if node_j in df_delta_all.index:
            U_global[6:12] = df_delta_all.loc[node_j, dof_mapping].to_numpy().astype(float)
            
        node_j_loads = df_loads_total[df_loads_total['node'] == node_j]
        if not node_j_loads.empty:
            F_fixed_global[6:12] = node_j_loads[load_cols].iloc[0].to_numpy().astype(float)

        # ======================================================================
        # STEP B: EVALUATE ALGEBRAIC EQUATIONS VIA MATRIX MULTIPLICATION
        # ======================================================================
        # {local displacement matrix} = [rotation matrix] * {global displacement matrix}
        u_local = r_matrix @ U_global
        
        # {local fixed end forces} = [rotation matrix] * {global fixed end forces}
        f_fixed_local = r_matrix @ F_fixed_global
        
        # {local member forces} = [local stiffness matrix] * {local displacement matrix} - {local fixed end forces}
        f_local = (k_local @ u_local) - f_fixed_local

        # ======================================================================
        # STEP C: STRUCTURE INTERMEDIATE COMPLIANCE ARRAYS INTO ELEMENT DATAFRAMES
        # ======================================================================
        # Store as 12x1 structural dataframes matching row/column layout definitions
        list_u_local_df[elem_id] = pd.DataFrame(
            u_local.reshape(-1, 1), index=force_rows, columns=['Local Disp']
        )
        
        list_f_fixed_local_df[elem_id] = pd.DataFrame(
            f_fixed_local.reshape(-1, 1), index=force_rows, columns=['Local Fixed Load']
        )
        
        list_f_local_df[elem_id] = pd.DataFrame(
            f_local.reshape(-1, 1), index=force_rows, columns=['Local Force']
        )

        # ======================================================================
        # STEP D: STORE ELEMENT LOCAL FORCES FOR SUMMARY
        # ======================================================================
        # Instead of accumulating per node, save the nodes and all 12 local forces
        row_data = [node_i, node_j] + f_local.flatten().tolist()
        element_force_summary.append(row_data)

    # ======================================================================
    # STEP E: SYNTHESIZE STRUCTURAL LOCALFORCE SUMMARY DATAFRAME
    # ======================================================================
    summary_cols = [
        'Node_I', 'Node_J',
        "fx'i", "fy'i", "fz'i", "mx'i", "my'i", "mz'i",
        "fx'j", "fy'j", "fz'j", "mx'j", "my'j", "mz'j"
    ]
    
    df_localforce_summary = pd.DataFrame(
        element_force_summary, 
        index=pd.Index(element_ids, name='Member_ID'), 
        columns=summary_cols
    )

    return list_u_local_df, list_f_fixed_local_df, list_f_local_df, df_localforce_summary

list_u_local_df, list_f_fixed_local_df, list_f_local_df, df_localforce_summary = evaluate_local_member_forces (
    df_geom=df_geom, 
    list_k_local=list_k_local, 
    list_r=list_r, 
    df_delta_all=df_delta_all, 
    df_loads_total=df_loads_total)

In [60]:
# List the columns you want to format
target_columns = [ "fx'i", "fy'i", "fz'i", "mx'i", "my'i", "mz'i", "fx'j", "fy'j", "fz'j", "mx'j", "my'j", "mz'j"]

# Apply to just those columns
df_localforce_summary[target_columns] = df_localforce_summary[target_columns].map(format_engineering)

df_localforce_summary

,Node_I,Node_J,fx'i,fy'i,fz'i,mx'i,my'i,mz'i,fx'j,fy'j,fz'j,mx'j,my'j,mz'j
Member_ID,,,,,,,,,,,,,,
1,1,4,124.913e+00,-40.965e-03,53.673e-03,66.665e+00,3.672e+03,5.468e+03,-105.425e+00,40.965e-03,-53.673e-03,-66.665e+00,-512.935e+00,-1.843e+03
2,2,5,173.398e+00,-2.161e+00,-896.099e-03,6.655e+00,721.588e+00,647.089e+00,-150.406e+00,2.161e+00,896.099e-03,-6.655e+00,-485.840e+00,-202.694e+00
3,3,6,111.584e+00,-2.291e+00,-703.992e-03,-39.207e+00,-580.199e+00,-1.214e+03,-92.912e+00,2.291e+00,703.992e-03,39.207e+00,-495.413e+00,1.465e+03
4,24,28,256.594e+00,448.490e-03,875.916e-03,-31.605e+00,2.054e+03,5.002e+03,-231.154e+00,-448.490e-03,-875.916e-03,31.605e+00,-128.266e+00,-1.865e+03
5,25,29,368.705e+00,-2.012e+00,-309.186e-03,51.457e+00,-2.591e+03,-259.533e+00,-340.625e+00,2.012e+00,309.186e-03,-51.457e+00,-184.999e+00,-294.174e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,90,110,-7.709e+00,18.147e+00,825.207e-03,4.211e+03,-1.036e+03,279.132e+00,7.709e+00,11.523e+00,-825.207e-03,3.411e+03,-1.038e+03,2.539e+03
189,110,93,-5.590e+00,3.876e+00,2.277e+00,6.824e+03,-1.555e+03,-8.641e+03,5.590e+00,24.305e+00,-2.277e+00,-2.463e+03,-2.521e+03,-9.008e+03
190,84,111,-1.961e+00,23.124e+00,4.291e+00,-1.153e+03,-4.868e+03,32.225e+03,1.961e+00,-8.139e+00,-4.291e+00,-2.640e+03,-3.126e+03,5.769e+03


In [61]:
df_localforce_summary.to_clipboard()

In [62]:
    # Extract Node I coordinates
nodes_i = df_geom[['Node_I', 'Xi', 'Yi', 'Zi']].rename(
    columns={'Node_I': 'node', 'Xi': 'X', 'Yi': 'Y', 'Zi': 'Z'}
)
# Extract Node J coordinates
nodes_j = df_geom[['Node_J', 'Xj', 'Yj', 'Zj']].rename(
    columns={'Node_J': 'node', 'Xj': 'X', 'Yj': 'Y', 'Zj': 'Z'}
)
# Combine and keep unique nodes
df_nodes = pd.concat([nodes_i, nodes_j]).drop_duplicates(subset=['node'])
df_supp = df_fixities.merge(df_nodes, on='node', how='inner')
df_supp

,node,x,y,z,x-rot,y-rot,z-rot,X,Y,Z
0,1,fixed,fixed,fixed,fixed,fixed,fixed,0,0,0
1,2,fixed,fixed,fixed,fixed,fixed,fixed,0,0,4500
2,3,fixed,fixed,fixed,fixed,fixed,fixed,0,0,8150
3,24,fixed,fixed,fixed,fixed,fixed,fixed,4800,0,0
4,25,fixed,fixed,fixed,fixed,fixed,fixed,4800,0,4500
5,26,fixed,fixed,fixed,fixed,fixed,fixed,4800,0,8150
6,27,fixed,fixed,fixed,fixed,fixed,fixed,4800,0,10500
7,46,fixed,fixed,fixed,fixed,fixed,fixed,11000,0,0
8,47,fixed,fixed,fixed,fixed,fixed,fixed,11000,0,4500
9,62,fixed,fixed,fixed,fixed,fixed,fixed,13000,0,4500


In [63]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def plot_interactive_structure(df_geom, df_loads_total, df_fixities):
    fig = go.Figure()

    # =========================================================================
    # 0. DATA PREP: EXTRACT NODES & FORCE STRING TYPES
    # =========================================================================
    # Extract Node coordinates
    nodes_i = df_geom[['Node_I', 'Xi', 'Yi', 'Zi']].rename(
        columns={'Node_I': 'node', 'Xi': 'X', 'Yi': 'Y', 'Zi': 'Z'}
    )
    nodes_j = df_geom[['Node_J', 'Xj', 'Yj', 'Zj']].rename(
        columns={'Node_J': 'node', 'Xj': 'X', 'Yj': 'Y', 'Zj': 'Z'}
    )
    df_nodes = pd.concat([nodes_i, nodes_j]).drop_duplicates(subset=['node'])

    # Create local copies to avoid altering your global dataframes
    df_loads_local = df_loads_total.copy()
    df_fixities_local = df_fixities.copy()

    # FORCE all 'node' columns to string format to prevent ValueError on merge
    df_nodes['node'] = df_nodes['node'].astype(str).str.strip()
    df_loads_local['node'] = df_loads_local['node'].astype(str).str.strip()
    df_fixities_local['node'] = df_fixities_local['node'].astype(str).str.strip()

    # Merge spatial coordinates into loads and fixities safely
    df_loads = df_loads_local.merge(df_nodes, on='node', how='left')
    df_supp = df_fixities_local.merge(df_nodes, on='node', how='inner')

    # =========================================================================
    # 1. TRACE: STRUCTURE GEOMETRY (LINES)
    # =========================================================================
    edge_x, edge_y, edge_z = [], [], []
    mid_x, mid_y, mid_z, elem_labels = [], [], [], []
    
    for _, row in df_geom.iterrows():
        edge_x.extend([row['Xi'], row['Xj'], None])
        edge_y.extend([row['Yi'], row['Yj'], None])
        edge_z.extend([row['Zi'], row['Zj'], None])
        
        mid_x.append((row['Xi'] + row['Xj']) / 2)
        mid_y.append((row['Yi'] + row['Yj']) / 2)
        mid_z.append((row['Zi'] + row['Zj']) / 2)
        elem_labels.append(f"E{int(row['Member_ID'])}")

    trace_elements = go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode='lines',
        line=dict(color='black', width=4),
        name='Elements',
        hoverinfo='none'
    )
    fig.add_trace(trace_elements)

    # =========================================================================
    # 2. TRACE: NODE LABELS & ELEMENT LABELS
    # =========================================================================
    trace_nodes = go.Scatter3d(
        x=df_nodes['X'], y=df_nodes['Y'], z=df_nodes['Z'],
        mode='markers+text',
        marker=dict(size=4, color='gray'),
        text=df_nodes['node'],
        textposition="top center",
        name='Nodes'
    )
    fig.add_trace(trace_nodes)

    trace_elem_labels = go.Scatter3d(
        x=mid_x, y=mid_y, z=mid_z,
        mode='text',
        text=elem_labels,
        textfont=dict(color='green', size=10),
        name='Element IDs'
    )
    fig.add_trace(trace_elem_labels)

    # =========================================================================
    # 3. TRACE: FIXITIES (SUPPORTS)
    # =========================================================================
    fixity_cols = ['x', 'y', 'z', 'x-rot', 'y-rot', 'z-rot']
    
    # Force columns to lowercase strings to cleanly match the word "fixed"
    for col in fixity_cols:
        df_supp[col] = df_supp[col].astype(str).str.lower().str.strip()
        
    # Keep only nodes where at least ONE degree of freedom says 'fixed'
    supports = df_supp[(df_supp[fixity_cols] == 'fixed').any(axis=1)]
    
    trace_supports = go.Scatter3d(
        x=supports['X'], y=supports['Y'], z=supports['Z'],
        mode='markers',
        marker=dict(symbol='diamond', size=8, color='orange', line=dict(color='black', width=1)),
        name='Supports'
    )
    fig.add_trace(trace_supports)

    # =========================================================================
    # 4. TRACES: FORCES AND MOMENTS (CONES/ARROWS)
    # =========================================================================
    # Ensure numeric types to prevent drawing failures
    load_cols = ['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']
    for col in load_cols:
        df_loads[col] = pd.to_numeric(df_loads[col], errors='coerce').fillna(0)

    # Filter forces
    forces = df_loads[(df_loads['Fx'] != 0) | (df_loads['Fy'] != 0) | (df_loads['Fz'] != 0)]
    trace_forces = go.Cone(
        x=forces['X'], y=forces['Y'], z=forces['Z'],
        u=forces['Fx'], v=forces['Fy'], w=forces['Fz'],
        sizemode="scaled", sizeref=0.5, anchor="tip", # 'scaled' auto-adjusts to building size
        colorscale=[[0, 'blue'], [1, 'blue']], showscale=False,
        name='Forces (Blue)'
    )
    fig.add_trace(trace_forces)

    # Filter moments
    moments = df_loads[(df_loads['Mx'] != 0) | (df_loads['My'] != 0) | (df_loads['Mz'] != 0)]
    trace_moments = go.Cone(
        x=moments['X'], y=moments['Y'], z=moments['Z'],
        u=moments['Mx'], v=moments['My'], w=moments['Mz'],
        sizemode="scaled", sizeref=0.5, anchor="tip", 
        colorscale=[[0, 'red'], [1, 'red']], showscale=False,
        name='Moments (Red)'
    )
    fig.add_trace(trace_moments)

    # =========================================================================
    # 5. INTERACTIVE BUTTONS (UPDATEMENUS)
    # =========================================================================
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.0, y=1.1,
                showactive=True,
                buttons=list([
                    dict(label="Show All", method="update",
                         args=[{"visible": [True, True, True, True, True, True]}]),
                    dict(label="Hide Nodes", method="update",
                         args=[{"visible": [True, False, True, True, True, True]}]),
                    dict(label="Hide Elem IDs", method="update",
                         args=[{"visible": [True, True, False, True, True, True]}]),
                    dict(label="Hide Loads", method="update",
                         args=[{"visible": [True, True, True, True, False, False]}])
                ]),
            )
        ],
        scene=dict(
            xaxis=dict(title='X Axis', showgrid=True),
            yaxis=dict(title='Y Axis', showgrid=True),
            zaxis=dict(title='Z Axis', showgrid=True),
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title="Interactive 3D Structural Model"
    )

    fig.show(renderer="browser")

# Execution
# plot_interactive_structure(df_geom, df_loads_total, df_fixities)

In [64]:
plot_interactive_structure(df_geom=df_geom, df_loads_total=df_loads_total, df_fixities=df_fixities)

In [65]:
import os
import nbformat
from nbconvert import HTMLExporter
from weasyprint import HTML, CSS

def export_notebook_to_annex_e(notebook_path, output_pdf_path="Annex_E_Report.pdf"):
    print(f"Reading notebook: {notebook_path}...")
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_content = nbformat.read(f, as_version=4)
        
    # 1. Initialize the HTML Exporter from nbconvert
    html_exporter = HTMLExporter()
    # Optional: exclude input or output prompts if you want a cleaner look
    # html_exporter.exclude_input_prompt = True
    # html_exporter.exclude_output_prompt = True
    
    # 2. Convert the notebook to standard HTML string
    (body, resources) = html_exporter.from_notebook_node(notebook_content)
    
    # 3. Define the strict CSS Paged Media rules for Annex E
    # This automatically builds the header, footer page counts, and handles line wrapping
    custom_css = """
    @page {
        size: A4 portrait;
        margin: 20mm 15mm 20mm 15mm;
        
        @top-right {
            content: "Annex E. Python Code";
            font-family: 'Courier New', Courier, monospace;
            font-size: 8.5pt;
            color: #333333;
            font-weight: bold;
        }
        
        @bottom-right {
            /* WeasyPrint dynamically computes page and pages variables natively */
            content: "Page E." counter(page) " of E." counter(pages);
            font-family: 'Courier New', Courier, monospace;
            font-size: 8.5pt;
            color: #333333;
        }
    }
    
    /* Global formatting fixes for high-quality printing */
    body {
        font-family: 'Courier New', Courier, monospace !important;
        font-size: 9pt !important;
    }
    
    /* Force long lines of code and text inputs to wrap cleanly instead of clipping horizontally */
    pre, code, .highlight, .input_area, .output_text pre {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-all !important;
    }
    
    /* Ensure markdown headers inside the notebook look clean and intentional */
    h1, h2, h3, h4 {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif !important;
        page-break-after: avoid !important;
    }
    
    /* Avoid cutting an atomic code cell cleanly across two pages if possible */
    .cell {
        page-break-inside: auto !important;
    }
    .input_area, .output_wrapper {
        page-break-inside: avoid !important;
    }
    """
    
    # 4. Inject our custom CSS block directly into the HTML body header
    styled_html = body.replace("</head>", f"<style>{custom_css}</style></head>")
    
    print("Compiling styled HTML into final PDF...")
    # 5. Compile to PDF via WeasyPrint
    HTML(string=styled_html).write_pdf(output_pdf_path)
    print(f"Successfully generated: {output_pdf_path}")

# ==============================================================================
# EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # Replace with your actual notebook file name
    target_notebook = "ce201_proj_code.ipynb" 
    
    if os.path.exists(target_notebook):
        export_notebook_to_annex_e(target_notebook, "Annex_E_Python_Code.pdf")
    else:
        print(f"Error: Could not find notebook file '{target_notebook}' in the current directory.")

Reading notebook: ce201_proj_code.ipynb...
Compiling styled HTML into final PDF...
Successfully generated: Annex_E_Python_Code.pdf
